# Lesson 1.9: Advanced Data Wrangling & Analysis

## Lesson Overview

This workshop transforms basic Python knowledge into professional data manipulation skills. We follow the "Extract-Transform-Analyze" workflow across 4 distinct sections:

1.  **Part 1: Financial Time Series & Window Functions**
    * Handling Datetime objects and indexing
    * Resampling and Frequency conversion
    * Window functions (Rolling means)
    * Covariance and Correlation
2.  **Part 2: Data Wrangling (Merge & Reshape)**
    * Merging datasets (Inner, Outer, Left, Right joins)
    * Reshaping data: Melt and Pivot
3.  **Part 3: Aggregation & Reporting**
    * GroupBy mechanics (Split-Apply-Combine)
    * Pivot Tables and Cross-Tabulations
4.  **Part 4: Advanced Toolkit (Optional/Deep Dive)**
    * Hierarchical Indexing (MultiIndex)
    * Concatenation
    * Stacking/Unstacking
    * Advanced GroupBy: Apply and Transform

---

**Setup:** Import necessary libraries.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Part 1: Financial Time Series & Window Functions

**Learning Objective:** Handle data where the *order* matters (Time Series). We will learn to convert strings to dates, handle missing business days, smoothing volatile data, and analyzing stock correlations.

## 1.1 Handling Date Time Data

Pandas is oriented towards working with arrays of dates, whether used as an axis index or a column.

The `to_datetime` method parses may different kinds of date representations:

In [2]:
dates = ["2011-07-06 12:00:00", "2011-08-06 00:00:00"]

pd.to_datetime(dates)

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00'], dtype='datetime64[ns]', freq=None)

It uses `NaT` (Not a Time) as null values for datetime data.

In [5]:
dates + [None]

['2011-07-06 12:00:00', '2011-08-06 00:00:00', None]

In [4]:
idx = pd.to_datetime(dates + [None])
idx

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00', 'NaT'], dtype='datetime64[ns]', freq=None)

**Gemini**

1. dates + [None]

This expression takes your existing list (dates) and combines it with a list containing a single None value. This is a common idiom to ensure that if a dataset contains missing entries (represented by Python's None), pandas knows how to handle them.

2. pd.to_datetime(...)

This is the core parsing engine. It takes the combined list and translates it into a standard, machine-readable format:Valid date strings (e.g., "2026-05-01") are converted into pandas Timestamp objects.The None value is automatically converted into a NaT (Not a Time), which is pandas' equivalent of a missing timestamp or NaN.

3. idx

Because you passed an array-like list (a sequence) to pd.to_datetime, the result is stored as a DatetimeIndex. This is a specialized, highly optimized index type in pandas specifically designed for time series analysis.

What you can do with idx:

Because it is a DatetimeIndex, you can now perform powerful time-based operations, such as:

Checking for missing values: idx.isna() returns array([False, False, True]).

Resampling, frequency calculations, or slicing by date ranges.

In [6]:
pd.isna(idx)

array([False, False,  True])

Standard Python uses the `datetime` module to handle date and time data. Pandas has a `Timestamp` object that is similar to the `datetime` object. 

If you use `datetime` objects as index to a Series or DataFrame, Pandas will automatically convert them to `DatetimeIndex` objects.

In [7]:
dates = [datetime(2011, 1, 2), datetime(2011, 1, 5), datetime(2011, 1, 7), 
         datetime(2011, 1, 8), datetime(2011, 1, 10), datetime(2011, 1, 12)]

ts = pd.Series(np.random.standard_normal(6), index=dates)
ts

2011-01-02    1.145166
2011-01-05    1.464979
2011-01-07   -1.821678
2011-01-08    0.596688
2011-01-10   -0.240071
2011-01-12   -0.198817
dtype: float64

In [8]:
ts.index

DatetimeIndex(['2011-01-02', '2011-01-05', '2011-01-07', '2011-01-08',
               '2011-01-10', '2011-01-12'],
              dtype='datetime64[ns]', freq=None)

Like other Series, arithmetic operations between differently indexed time series automatically align on the dates:

In [9]:
ts

2011-01-02    1.145166
2011-01-05    1.464979
2011-01-07   -1.821678
2011-01-08    0.596688
2011-01-10   -0.240071
2011-01-12   -0.198817
dtype: float64

In [10]:
ts[::2]

2011-01-02    1.145166
2011-01-07   -1.821678
2011-01-10   -0.240071
dtype: float64

In [11]:
# [::2] selects every second element
ts + ts[::2]

2011-01-02    2.290332
2011-01-05         NaN
2011-01-07   -3.643356
2011-01-08         NaN
2011-01-10   -0.480142
2011-01-12         NaN
dtype: float64

NaN because ts[::2] has no values for the dates 2011-01-05, 2011-01-08 and 2011-01-12. If a number doesn't exist, the output is NaN.

### Indexing & Slicing

You can index by passing a `datetime`, `Timestamp` or `string` that is interpretable as a date:

In [12]:
ts

2011-01-02    1.145166
2011-01-05    1.464979
2011-01-07   -1.821678
2011-01-08    0.596688
2011-01-10   -0.240071
2011-01-12   -0.198817
dtype: float64

In [13]:
ts[datetime(2011, 1, 7)]

-1.821677860447044

In [14]:
ts[pd.Timestamp("2011-01-07")]

-1.821677860447044

In [15]:
ts["2011-01-07"]

-1.821677860447044

You can even specify the year or year-month strings to slice a range of data. This is very powerful for quick analysis.

In [16]:
# date_range generate an array of dates
longer_ts = pd.Series(np.random.standard_normal(1000), 
                      index=pd.date_range("2000-01-01", periods=1000))

longer_ts

2000-01-01    1.240062
2000-01-02   -0.302071
2000-01-03    1.491564
2000-01-04   -0.205977
2000-01-05   -1.124395
                ...   
2002-09-22    0.475791
2002-09-23    0.559806
2002-09-24    0.393127
2002-09-25   -0.463674
2002-09-26    0.160637
Freq: D, Length: 1000, dtype: float64

In [17]:
# Select all data from 2001
longer_ts["2001"].head()

2001-01-01    0.299116
2001-01-02    0.357348
2001-01-03    0.182603
2001-01-04    1.103279
2001-01-05    0.586106
Freq: D, dtype: float64

With the .head(), default is 5 entries.

In [18]:
longer_ts["2001"]

2001-01-01    0.299116
2001-01-02    0.357348
2001-01-03    0.182603
2001-01-04    1.103279
2001-01-05    0.586106
                ...   
2001-12-27   -2.038684
2001-12-28   -1.960175
2001-12-29    0.558111
2001-12-30   -0.604633
2001-12-31    1.759707
Freq: D, Length: 365, dtype: float64

In [19]:
# Select all data from May 2001
longer_ts["2001-05"].head()

2001-05-01    0.854570
2001-05-02   -1.186840
2001-05-03    0.614341
2001-05-04   -0.087716
2001-05-05    0.872011
Freq: D, dtype: float64

In [20]:
len(longer_ts["2001-05"])

31

In [21]:
len(longer_ts["2001-05":])

514

In [22]:
# Slicing with strings
longer_ts["2001-05":].head()

2001-05-01    0.854570
2001-05-02   -1.186840
2001-05-03    0.614341
2001-05-04   -0.087716
2001-05-05    0.872011
Freq: D, dtype: float64

> **🛠️ Exercise:** Use `date_range` to generate a Series of random values from 1-31st January 2023. Then slice the Series to return data from 5-15th January.

In [23]:
# Use `date_range` to generate a Series of random values from 1-31st January 2023.
Exercise_ts = pd.Series(np.random.standard_normal(31), 
                      index=pd.date_range("2023-01-01", periods=31))
Exercise_ts.head()

2023-01-01    0.336472
2023-01-02    0.350830
2023-01-03    1.050704
2023-01-04    0.555303
2023-01-05   -0.922156
Freq: D, dtype: float64

In [24]:
# Then slice the Series to return data from 5-15th January.
Exercise_ts["2023-01-05":"2023-01-15"]

2023-01-05   -0.922156
2023-01-06   -0.331914
2023-01-07   -0.262222
2023-01-08    1.445506
2023-01-09   -0.108527
2023-01-10   -0.551691
2023-01-11   -0.638050
2023-01-12    0.577143
2023-01-13    0.693617
2023-01-14   -0.116626
2023-01-15   -0.628604
Freq: D, dtype: float64

## 1.2 Real-World Application: Stock Market Data

Let's load stock prices (AAPL, GOOG, IBM, MSFT) and trade volumes. We use `.read_pickle` here as it preserves the datetime objects native to the file.

**Gemini:**

How to convert and store a pickle file.

*Option 1: Using Pandas (Easiest & Fastest)*

The Pandas library has built-in methods for both CSV and pickle operations. This is the standard way to handle tabular data.pythonimport pandas as pd

1. Read the CSV file into a DataFrame
df = pd.read_csv('data.csv')

2. Save (pickle) the DataFrame to a file
df.to_pickle('data.pkl')

To read it back later:
  df = pd.read_pickle('data.pkl')

*Option 2: Using the Standard pickle and csv Libraries*

If you prefer not to use external libraries, you can use Python's built-in csv module to read the file into a list and then pickle.dump() to save it.
pythonimport csv
import pickle

1. Read the CSV data into a list
with open('data.csv', 'r') as f:
    reader = csv.reader(f)
    data_list = list(reader)

2. Pickle the list into a binary file
with open('data.pkl', 'wb') as f:
    pickle.dump(data_list, f)

* To read it back later:
with open('data.pkl', 'rb') as f:
    data = pickle.load(f)

**Key Differences**
- Performance: Pickling is significantly faster for large datasets than re-parsing a CSV every time you run your script.
- File Format: CSVs are plain text (human-readable), while pickle files are binary (computer-readable only).

Security: * Never "unpickle" a file from an untrusted source, as it can execute malicious code.

In [25]:
price = pd.read_pickle("../data/yahoo_price.pkl")
volume = pd.read_pickle("../data/yahoo_volume.pkl")

In [26]:
type(price)

pandas.core.frame.DataFrame

In [27]:
price.head()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-04,27.990226,313.062468,113.304536,25.884104
2010-01-05,28.038618,311.683844,111.935822,25.892466
2010-01-06,27.592626,303.826685,111.208683,25.733566
2010-01-07,27.541619,296.753749,110.823732,25.465944
2010-01-08,27.724725,300.709808,111.935822,25.641571


In [28]:
volume.head()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-04,123432400,3927000,6155300,38409100
2010-01-05,150476200,6031900,6841400,49749600
2010-01-06,138040000,7987100,5605300,58182400
2010-01-07,119282800,12876600,5840600,50559700
2010-01-08,111902700,9483900,4197200,51197400


### Inspecting the Index
Notice the index is a `DatetimeIndex`.

In [29]:
price.index

DatetimeIndex(['2010-01-04', '2010-01-05', '2010-01-06', '2010-01-07',
               '2010-01-08', '2010-01-11', '2010-01-12', '2010-01-13',
               '2010-01-14', '2010-01-15',
               ...
               '2016-10-10', '2016-10-11', '2016-10-12', '2016-10-13',
               '2016-10-14', '2016-10-17', '2016-10-18', '2016-10-19',
               '2016-10-20', '2016-10-21'],
              dtype='datetime64[ns]', name='Date', length=1714, freq=None)

We can access attributes like `day_of_week` directly:

In [30]:
price.index.day_of_week

Int64Index([0, 1, 2, 3, 4, 0, 1, 2, 3, 4,
            ...
            0, 1, 2, 3, 4, 0, 1, 2, 3, 4],
           dtype='int64', name='Date', length=1714)

**Note**
0 is Monday. 0 to 4 - stock exchanges operate from Mondays to Fridays

In [32]:
price.index.month

Int64Index([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
            ...
            10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
           dtype='int64', name='Date', length=1714)

If the datetime is in a column instead of the index, you can use the `dt` accessor to access the datetime properties.

In [33]:
price_reindex = price.reset_index()
price_reindex.head()

,Date,AAPL,GOOG,IBM,MSFT
0,2010-01-04,27.990226,313.062468,113.304536,25.884104
1,2010-01-05,28.038618,311.683844,111.935822,25.892466
2,2010-01-06,27.592626,303.826685,111.208683,25.733566
3,2010-01-07,27.541619,296.753749,110.823732,25.465944
4,2010-01-08,27.724725,300.709808,111.935822,25.641571


Purpose of reindex : so we can use the date column in a function

After reindexing, the index is the leftmost column with 0 ~ 4

In [34]:
price_reindex.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1714 entries, 0 to 1713
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1714 non-null   datetime64[ns]
 1   AAPL    1714 non-null   float64       
 2   GOOG    1714 non-null   float64       
 3   IBM     1714 non-null   float64       
 4   MSFT    1714 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 67.1 KB


if any Dtype is a string or na, then the answer will be NaN.

In [35]:
price_reindex["Date"].dt.day_name().head()

0       Monday
1      Tuesday
2    Wednesday
3     Thursday
4       Friday
Name: Date, dtype: object

> **🛠️ Exercise:** Get the week of year from the date column and create a new column `week_of_year`.

In [36]:
# Get the week of year from the date column and create a new column `week_of_year`. My answer:
price_reindex["week_of_year"] = price_reindex["Date"].dt.isocalendar().week
price_reindex.head()

,Date,AAPL,GOOG,IBM,MSFT,week_of_year
0,2010-01-04,27.990226,313.062468,113.304536,25.884104,1
1,2010-01-05,28.038618,311.683844,111.935822,25.892466,1
2,2010-01-06,27.592626,303.826685,111.208683,25.733566,1
3,2010-01-07,27.541619,296.753749,110.823732,25.465944,1
4,2010-01-08,27.724725,300.709808,111.935822,25.641571,1


In [38]:
# Get the week of year from the date column and create a new column `week_of_year`. Yibin's answer:
price_reindex['week_of_year'] = price_reindex["Date"].dt.weekofyear
price_reindex.head()

/tmp/ipykernel_9229/4149890348.py:2: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated. Please use Series.dt.isocalendar().week instead.
  price_reindex['week_of_year'] = price_reindex["Date"].dt.weekofyear


,Date,AAPL,GOOG,IBM,MSFT,week_of_year
0,2010-01-04,27.990226,313.062468,113.304536,25.884104,1
1,2010-01-05,28.038618,311.683844,111.935822,25.892466,1
2,2010-01-06,27.592626,303.826685,111.208683,25.733566,1
3,2010-01-07,27.541619,296.753749,110.823732,25.465944,1
4,2010-01-08,27.724725,300.709808,111.935822,25.641571,1


Don't panic when we see warning. Only act when we have errors.

## 1.3 Resampling (Frequency Conversion)

As you can see from above, the dates are on business days. If you want to change the frequency to calendar days (e.g. including weekends), we use `resample`.

This introduces missing data (NaN) for weekends.

In [40]:
price_resampled = price.resample('D').asfreq()
price_resampled.head(10)

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-04,27.990226,313.062468,113.304536,25.884104
2010-01-05,28.038618,311.683844,111.935822,25.892466
2010-01-06,27.592626,303.826685,111.208683,25.733566
2010-01-07,27.541619,296.753749,110.823732,25.465944
2010-01-08,27.724725,300.709808,111.935822,25.641571
2010-01-09,NaN,NaN,NaN,NaN
2010-01-10,NaN,NaN,NaN,NaN
2010-01-11,27.480148,300.255255,110.763844,25.315406
2010-01-12,27.167562,294.945572,111.644958,25.148142


**Gemini**

This code snippet changes the frequency of a time-series dataset to daily intervals ('D') and fills in any missing dates with empty values (NaN).

1. price.resample('D')

- This targets the index of your price DataFrame, which must be a DatetimeIndex.
- It groups your data into a daily frequency ('D').
- This creates a temporary "resampler" object but does not change the data yet.

2. .asfreq()

- This applies the new daily frequency to your data.
- If a date was missing from the original data (like a weekend or holiday), it creates a new row for that day.
- It fills the newly created rows with NaN (Not a Number) because no data existed for those dates.

3. price_resampled.head(10)

- This displays the first 10 rows of the newly adjusted DataFrame so you can inspect the results.

*Why use this?*

It is a crucial step before doing time-series mathematical modeling. Many models require a continuous timeline without any gaps in the dates.

**Forward Fill (`ffill`)**: If you want to fill the na values with the most recent value (common in finance - Saturday price is same as Friday close), you can use the `.ffill()` method.

In [41]:
price_resampled = price.resample('D').ffill()
price_resampled.head(10)

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-04,27.990226,313.062468,113.304536,25.884104
2010-01-05,28.038618,311.683844,111.935822,25.892466
2010-01-06,27.592626,303.826685,111.208683,25.733566
2010-01-07,27.541619,296.753749,110.823732,25.465944
2010-01-08,27.724725,300.709808,111.935822,25.641571
2010-01-09,27.724725,300.709808,111.935822,25.641571
2010-01-10,27.724725,300.709808,111.935822,25.641571
2010-01-11,27.480148,300.255255,110.763844,25.315406
2010-01-12,27.167562,294.945572,111.644958,25.148142


If you want to resample to a **lower frequency** (e.g. Monthly 'MS' - Month Start) you need to provide an aggregation method (like `mean`):

In [42]:
price_resampled = price.resample('MS').mean() # MS means month start
price_resampled.head()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-01,27.166942,289.012006,110.364037,25.212407
2010-02-01,26.000370,267.179857,107.496919,23.767984
2010-03-01,29.219760,280.235245,109.849295,24.584058
2010-04-01,32.847556,278.253415,111.324726,25.646245
2010-05-01,32.888483,248.418509,109.690449,23.670255


> **🛠️ Exercise:** Resample price to `yearly` (start of year) frequency, use `sum` as aggregation function.

In [44]:
# Resample price to `yearly` (start of year) frequency, use `sum` as aggregation function.
price_resampled = price.resample('YS').sum() # YS means year start
price_resampled.head()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-01,8564.125188,67421.204377,28722.464346,5757.023922
2011-01-01,11997.187091,71619.248624,37895.630612,5675.814809
2012-01-01,18880.993859,80271.925770,43947.570175,6621.497457
2013-01-01,15929.515474,111303.472578,44524.331233,7498.810042
2014-01-01,22293.854902,141458.282752,42727.132433,10081.875149


In [45]:
price.resample('Y').sum()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-12-31,8564.125188,67421.204377,28722.464346,5757.023922
2011-12-31,11997.187091,71619.248624,37895.630612,5675.814809
2012-12-31,18880.993859,80271.925770,43947.570175,6621.497457
2013-12-31,15929.515474,111303.472578,44524.331233,7498.810042
2014-12-31,22293.854902,141458.282752,42727.132433,10081.875149
2015-12-31,29501.178958,151820.449636,37431.251170,11389.086467
2016-12-31,20810.039594,150038.480286,29879.087154,10904.541111


**Gemini**
'Y' means end of year; YS means start of year.

price_resampled = price.resample('YS').sum()

In a stock context, price.resample('YS').sum() calculates the sum of all daily (or higher frequency) closing prices within each calendar year, starting from January 1st.

*What the Summation Value Means*

The sum represents the cumulative total of daily prices, which usually does not correspond to a standard financial metric like total returns or annual revenue.

Average Yearly Price: If you divide this sum by the number of trading days in that year, you get the average daily price."

Liquidity Pulse" (If applied to Volume): While less useful for price, if this formula is applied to volume (volume.resample('YS').sum()), it represents total trading activity.

Important Considerations

- 'YS' (Year Start): Specifies that the year starts on January 1st rather than the first trading day.
- Summation vs. Last: In finance, it is more common to use .last() (year-end price) to calculate annual returns, rather than .sum().

## 1.4 Window Functions (Moving Averages)

You can apply functions evaluated over a sliding window using the `rolling` method.

For example, to compute the 30-day moving average for Apple price:

In [46]:
price["AAPL"].rolling(30).mean().tail()

Date
2016-10-17    112.730333
2016-10-18    113.056000
2016-10-19    113.348000
2016-10-20    113.732666
2016-10-21    114.181667
Name: AAPL, dtype: float64

**Gemini**
This code snippet calculates the 30-day moving average of Apple (AAPL) stock prices and displays the most recent 5 calculations.

1. price["AAPL"]

- This selects the "AAPL" column from your price DataFrame.
- It extracts just the Apple stock price data as a pandas Series.

2. .rolling(30)
- This creates a moving "window" of the last 30 consecutive rows (days) in the data.
- As you move down the rows, the 30-day window shifts forward by one day, dropping the oldest day and adding the newest.

3. .mean()

- This calculates the average price of the 30 data points inside that shifting window.
- Note: The first 29 rows in your dataset will output NaN (Not a Number) because they do not have enough prior days to make a full 30-day window.

4. .tail()

- This limits the final output to just the last 5 rows of the calculated series.
- It allows you to see the most recent moving average trends without cluttering your screen with the entire history.

In [48]:
price["AAPL"].rolling(30).mean().head(50)

Date
2010-01-04          NaN
2010-01-05          NaN
2010-01-06          NaN
2010-01-07          NaN
2010-01-08          NaN
2010-01-11          NaN
2010-01-12          NaN
2010-01-13          NaN
2010-01-14          NaN
2010-01-15          NaN
2010-01-19          NaN
2010-01-20          NaN
2010-01-21          NaN
2010-01-22          NaN
2010-01-25          NaN
2010-01-26          NaN
2010-01-27          NaN
2010-01-28          NaN
2010-01-29          NaN
2010-02-01          NaN
2010-02-02          NaN
2010-02-03          NaN
2010-02-04          NaN
2010-02-05          NaN
2010-02-08          NaN
2010-02-09          NaN
2010-02-10          NaN
2010-02-11          NaN
2010-02-12          NaN
2010-02-16    26.645276
2010-02-17    26.595315
2010-02-18    26.545397
2010-02-19    26.504852
2010-02-22    26.460558
2010-02-23    26.395512
2010-02-24    26.354313
2010-02-25    26.329376
2010-02-26    26.303088
2010-03-01    26.301169
2010-03-02    26.313899
2010-03-03    26.289006
2010-03-04 

without .tail(), MA for first 30 days is NaN because lack of value for the last 30 days to calculate the rolling moving average.
.tail() returns the last elements, five by default.

By default, rolling functions require all of the values in the window to be non-NA. This behavior can be changed to account for missing data and, especially at the beginning of the time series.

In [49]:
price["AAPL"].rolling(30, min_periods=3).mean().head(50)

Date
2010-01-04          NaN
2010-01-05          NaN
2010-01-06    27.873823
2010-01-07    27.790772
2010-01-08    27.777563
2010-01-11    27.727994
2010-01-12    27.647932
2010-01-13    27.635787
2010-01-14    27.608612
2010-01-15    27.541096
2010-01-19    27.594173
2010-01-20    27.602327
2010-01-21    27.572405
2010-01-22    27.450347
2010-01-25    27.390950
2010-01-26    27.362438
2010-01-27    27.352205
2010-01-28    27.280694
2010-01-29    27.166942
2010-02-01    27.082025
2010-02-02    27.012234
2010-02-03    26.968821
2010-02-04    26.888355
2010-02-05    26.833177
2010-02-08    26.775403
2010-02-09    26.732486
2010-02-10    26.687565
2010-02-11    26.662434
2010-02-12    26.646749
2010-02-16    26.645276
2010-02-17    26.595315
2010-02-18    26.545397
2010-02-19    26.504852
2010-02-22    26.460558
2010-02-23    26.395512
2010-02-24    26.354313
2010-02-25    26.329376
2010-02-26    26.303088
2010-03-01    26.301169
2010-03-02    26.313899
2010-03-03    26.289006
2010-03-04 

In [50]:
price["AAPL"].rolling(30, min_periods=30).mean().head()

Date
2010-01-04   NaN
2010-01-05   NaN
2010-01-06   NaN
2010-01-07   NaN
2010-01-08   NaN
Name: AAPL, dtype: float64

**Gemini**

This code snippet calculates the 30-day moving average for Apple stock prices, explicitly forces the first 29 rows to be blank, and displays the very first 5 rows of the dataset.

1. price["AAPL"]This isolates the "AAPL" column from your DataFrame.

2. .rolling(30, min_periods=30)
- This creates the 30-row sliding window.min_periods=30 means the window must contain at least 30 valid data points to calculate a result.
- Note: Since 30 is already the default for a window of size 30, adding it here simply makes this strict requirement explicit in the code.

3. .mean()
- This calculates the mathematical average of the 30 rows in the window.

4. .head()
- This displays the first 5 rows of your dataset.

*What the Output Actually Looks Like:*

Because you asked for the .head() (the first 5 rows) of a 30-day moving average, all 5 rows will display NaN (Not a Number).

> **🛠️ Exercise:** Compute a 10-day moving average for `GOOG` with a min period of 5 days.

## 1.5 Covariance and Correlation

Covariance and correlation measure the relationship between two variables.

* **Covariance:** Measure of how much two random variables vary together. Hard to interpret magnitude.
* **Correlation:** Normalized measure (-1 to 1). 1 is perfect positive correlation, -1 is perfect negative.

In finance, we usually look at **Returns** (Percent Change), not raw prices.

In [51]:
returns = price.pct_change()
returns.tail()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2016-10-17,-0.000680,0.001837,0.002072,-0.003483
2016-10-18,-0.000681,0.019616,-0.026168,0.007690
2016-10-19,-0.002979,0.007846,0.003583,-0.002255
2016-10-20,-0.000512,-0.005652,0.001719,-0.004867
2016-10-21,-0.003930,0.003011,-0.012474,0.042096


Compute the correlation and covariance between the returns of `MSFT` and `IBM`:

In [52]:
print("Covariance:", returns["MSFT"].cov(returns["IBM"]))
print("Correlation:", returns["MSFT"].corr(returns["IBM"]))

Covariance: 8.870655479703549e-05
Correlation: 0.49976361144151155


You can also get the full (pair-wise) correlation or covariance matrix as a DataFrame:

In [53]:
returns.corr()

,AAPL,GOOG,IBM,MSFT
AAPL,1.000000,0.407919,0.386817,0.389695
GOOG,0.407919,1.000000,0.405099,0.465919
IBM,0.386817,0.405099,1.000000,0.499764
MSFT,0.389695,0.465919,0.499764,1.000000


**Gemini**

1. returns
- This is a DataFrame containing financial return data (usually daily percentage changes) for multiple assets or stocks (e.g., AAPL, MSFT, GOOG).

2. .corr()
- This automatically calculates the linear relationship between every pair of columns using the Pearson correlation coefficient by default.
- It outputs a grid (matrix) where both the rows and columns are the asset names.

You can also compute pair-wise correlations between a DataFrame’s columns or rows with another Series or DataFrame.

In [54]:
# Correlation of all companies against IBM
returns.corrwith(returns["IBM"])

AAPL    0.386817
GOOG    0.405099
IBM     1.000000
MSFT    0.499764
dtype: float64

In [55]:
# Correlation of returns against volume
returns.corrwith(volume)

AAPL   -0.075565
GOOG   -0.007067
IBM    -0.204849
MSFT   -0.092950
dtype: float64

---
# Part 2: Data Wrangling (Merge & Reshape)

**Learning Objective:** Combine data from different sources (SQL-style Joins) and reshape table layouts (Wide to Long) to prepare for analysis.

## 2.1 Merging (Joins)

`merge` connects rows in DataFrames based on one or more keys. This is equivalent to database `join` operations.

In [56]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"], 
                    "data1": pd.Series(range(7), dtype="Int64")})

df2 = pd.DataFrame({"key": ["a", "b", "d"], 
                    "data2": pd.Series(range(3), dtype="Int64")})

print("DF1 (Left):\n", df1)
print("\nDF2 (Right):\n", df2)

DF1 (Left):
   key  data1
0   b      0
1   b      1
2   a      2
3   c      3
4   a      4
5   a      5
6   b      6

DF2 (Right):
   key  data2
0   a      0
1   b      1
2   d      2


**Many-to-One Join:** `df1` has multiple rows labeled `a` and `b`, whereas `df2` has only one row for each value in the key column `key`.

The default is an **Inner Join** (intersection of keys).

In [57]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,b,6,1
3,a,2,0
4,a,4,0
5,a,5,0


It is good practice to specify the key explicitly:

In [58]:
pd.merge(df1, df2, on="key")

,key,data1,data2
0,b,0,1
1,b,1,1
2,b,6,1
3,a,2,0
4,a,4,0
5,a,5,0


If the column names are different in each object, you can specify them separately using `left_on` and `right_on`:

In [59]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"], 
                    "data1": pd.Series(range(7), dtype="Int64")})

df4 = pd.DataFrame({"rkey": ["a", "b", "d"], 
                    "data2": pd.Series(range(3), dtype="Int64")})

pd.merge(df3, df4, left_on="lkey", right_on="rkey")

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,b,6,b,1
3,a,2,a,0
4,a,4,a,0
5,a,5,a,0


### Join Types (Inner, Outer, Left, Right)

You can specify the other options via the `how` parameter.

In [60]:
# Outer Join: Union of keys. Fills missing with NaN
pd.merge(df1, df2, how="outer")

,key,data1,data2
0,b,0,1
1,b,1,1
2,b,6,1
3,a,2,0
4,a,4,0
5,a,5,0
6,c,3,<NA>
7,d,<NA>,2


In [65]:
# Outer Join with mismatched key names
pd.merge(df3, df4, left_on="lkey", right_on="rkey", how="outer")

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,b,6,b,1
3,a,2,a,0
4,a,4,a,0
5,a,5,a,0
6,c,3,NaN,<NA>
7,NaN,<NA>,d,2


**Many-to-Many Join:**

In [66]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], 
                    "data1": pd.Series(range(6), dtype="Int64")})

df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"], 
                    "data2": pd.Series(range(5), dtype="Int64")})

pd.merge(df1, df2, how="inner")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,b,5,1
5,b,5,3
6,a,2,0
7,a,2,2
8,a,4,0
9,a,4,2


> **🛠️ Exercise:** Merge `df1` and `df2` with a left join.

In [67]:
# Merge `df1` and `df2` with a left join.
pd.merge(df1, df2, how="left")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,c,3,<NA>
7,a,4,0
8,a,4,2
9,b,5,1


### Merging on Multiple Keys & Suffixes

To merge with multiple keys, pass a list of column names:

In [62]:
left = pd.DataFrame({"key1": ["foo", "foo", "bar"], 
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})

right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})

pd.merge(left, right, on=["key1", "key2"], how="outer")

,key1,key2,lval,rval
0,foo,one,1,4
1,foo,one,1,5
2,foo,two,2,<NA>
3,bar,one,3,6
4,bar,two,<NA>,7


If there are overlapping non-key column names, `merge` adds suffixes `_x` and `_y` by default. You can customize this:

In [64]:
pd.merge(left, right, on="key1")

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


In [66]:
pd.merge(left, right, on="key1", suffixes=("_left", "_right"))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


**Gemini**
This code snippet executes an inner join on two DataFrames (left and right) using key1 as the matching column, and renames any overlapping column names by adding custom suffixes.

1. pd.merge(left, right, on="key1")
- This blends the left DataFrame and the right DataFrame together.
- By default, how="inner" is applied, meaning rows are kept only if the value in key1 matches exactly in both tables.

2. suffixes=("_left", "_right")
- If both DataFrames contain columns with the exact same name (other than key1), pandas needs a way to tell them apart in the final output.
- This parameter automatically appends _left to the column coming from the left DataFrame, and _right to the column coming from the right DataFrame.

*Why use this?*
- It prevents pandas from using its default suffixes (_x and _y), which are generic and make code difficult to read. 
- Custom suffixes like _left/_right (or _broker/_exchange) clarify exactly where conflicting data originated.

### Merging on Index

If the merge key(s) is in the index, you can pass `left_index=True` or `right_index=True`.

In [68]:
left1 = pd.DataFrame({"key": ["a", "b", "a", "a", "b", "c"],
                      "value": pd.Series(range(6), dtype="Int64")})

right1 = pd.DataFrame({"group_val": [3.5, 7]}, index=["a", "b"])

pd.merge(left1, right1, left_on="key", right_index=True)

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0


DataFrame has a `join` method which performs a left join by default. It's a convenient shortcut for index-on-index merging.

Yibin: recommends to just use pd.merge

In [69]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


## 2.2 Reshaping and Pivoting

We often need to switch between **Wide Format** (Excel style, years as columns) and **Long Format** (Database style, one row per observation).

### Melt (Wide to Long)
Let's look at our stock price data. It is currently **Wide**.

In [70]:
# Reset index so Date is a column
price_reindex = price.reset_index()
price_reindex.head()

,Date,AAPL,GOOG,IBM,MSFT
0,2010-01-04,27.990226,313.062468,113.304536,25.884104
1,2010-01-05,28.038618,311.683844,111.935822,25.892466
2,2010-01-06,27.592626,303.826685,111.208683,25.733566
3,2010-01-07,27.541619,296.753749,110.823732,25.465944
4,2010-01-08,27.724725,300.709808,111.935822,25.641571


In [71]:
len(price_reindex)

1714

In [72]:
# Melt into Long format
melted = pd.melt(price_reindex, id_vars="Date")
melted

,Date,variable,value
0,2010-01-04,AAPL,27.990226
1,2010-01-05,AAPL,28.038618
2,2010-01-06,AAPL,27.592626
3,2010-01-07,AAPL,27.541619
4,2010-01-08,AAPL,27.724725
...,...,...,...
6851,2016-10-17,MSFT,57.220001
6852,2016-10-18,MSFT,57.660000
6853,2016-10-19,MSFT,57.529999
6854,2016-10-20,MSFT,57.250000


> **🛠️ Exercise:** Rerun `melt` and pass arguments such that the new columns are named `Company` and `Price` respectively.

### Pivot (Long to Wide)
Using `pivot`, we can reshape back to the original layout:

In [73]:
reshaped = melted.pivot(index='Date', columns='variable', values='value')
reshaped.head()

variable,AAPL,GOOG,IBM,MSFT
Date,,,,
2010-01-04,27.990226,313.062468,113.304536,25.884104
2010-01-05,28.038618,311.683844,111.935822,25.892466
2010-01-06,27.592626,303.826685,111.208683,25.733566
2010-01-07,27.541619,296.753749,110.823732,25.465944
2010-01-08,27.724725,300.709808,111.935822,25.641571


---
# Part 3: Aggregation & Reporting

**Learning Objective:** Summarize data using GroupBy, Custom Aggregations, and Pivot Tables to answer business questions.

## 3.1 Data Aggregation (GroupBy)

Data aggregation is the process of grouping data together and performing calculations on them.

In [74]:
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None], 
                   "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                   "data1" : np.random.standard_normal(7), 
                   "data2" : np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.522877,-0.309539
1,a,2,0.448135,0.581475
2,None,1,0.896047,-1.326059
3,b,2,-0.564818,0.130843
4,b,1,-0.095919,0.522011
5,a,<NA>,-0.652773,0.333030
6,None,1,0.314569,-0.359428


If you want to compute the mean for each unique value in `key1`:

In [76]:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.242505,0.201655
b,1.5,-0.330369,0.326427


It does not make sense to compute the mean for `key2` since it is a categorical variable and also serves as a key.

We can select the numeric columns to compute the mean for (after the `groupby` method):

In [77]:
df.groupby("key1")[["data1", "data2"]].mean()

,data1,data2
key1,,
a,-0.242505,0.201655
b,-0.330369,0.326427


Note that the following also works, since the returned result is a DataFrame, however it is less efficient as the selection/subset happens after the computation.

In [78]:
df.groupby("key1").mean()[["data1", "data2"]]

,data1,data2
key1,,
a,-0.242505,0.201655
b,-0.330369,0.326427


You can group by more than 1 column. There is a useful GroupBy method `size` which returns a Series containing group sizes.

In [79]:
df.groupby(['key1', 'key2']).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

You can also group by other `Series`/`array`/`list` with the same length:

In [85]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]

df["data1"].groupby([states, years]).mean()

CA  2005    0.104895
    2006   -0.235910
OH  2005   -1.184831
    2006   -0.324783
Name: data1, dtype: float64

> **🛠️ Exercise:** Group by `key1` and `key2` and compute the standard deviation.

## 3.2 Custom Aggregation

To use your own aggregation functions, pass any function that aggregates an array to the `aggregate` method or its short alias `agg`:

In [80]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [81]:
grouped = df.groupby("key1")
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,1.100908,0.891014
b,1,0.468899,0.391168


You can pass a list of functions, or function names (for built-in functions) to `aggregate`: 

In [82]:
grouped.agg([peak_to_peak, "mean", "std"])

key2                       data1                      \
     peak_to_peak mean       std peak_to_peak      mean       std   
key1                                                                
a               1  1.5  0.707107     1.100908 -0.242505  0.601628   
b               1  1.5  0.707107     0.468899 -0.330369  0.331562   

            data2                      
     peak_to_peak      mean       std  
key1                                   
a        0.891014  0.201655  0.459805  
b        0.391168  0.326427  0.276597

## 3.3 Pivot Tables

Pivot tables are used to summarize, sort, reorganize, group, count, total or average data. It allows its users to transform columns into rows and rows into columns.

We will use the `tips.csv` dataset.

In [83]:
tips = pd.read_csv("../data/tips.csv")

# add a column with the tip percentage
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


The default aggregation for `pivot_table` is mean.

In [84]:
tips.pivot_table(index=["day", "smoker"], values=["size", "tip", "tip_pct", "total_bill"])

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

You can put `smoker` in the table columns and `time` and `day` in the rows:

In [85]:
tips.pivot_table(index=["time", "day"], columns="smoker", 
                 values=["tip_pct", "size"])

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

Add partial totals by passing `margins=True`:

In [86]:
tips.pivot_table(index=["time", "day"], columns="smoker", 
                 values=["tip_pct", "size"], margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

To use other aggregation functions, pass it to the `aggfunc` keyword:

In [87]:
tips.pivot_table(index=["time", "smoker"], columns="day", 
                 values="tip_pct", aggfunc=len, margins=True)

day             Fri   Sat   Sun  Thur  All
time   smoker                             
Dinner No       3.0  45.0  57.0   1.0  106
       Yes      9.0  42.0  19.0   NaN   70
Lunch  No       1.0   NaN   NaN  44.0   45
       Yes      6.0   NaN   NaN  17.0   23
All            19.0  87.0  76.0  62.0  244

Use `fill_value` to fill missing values:

In [88]:
tips.pivot_table(index=["time", "smoker"], columns="day", 
                 values="tip_pct", aggfunc=len, margins=True, fill_value=0)

day            Fri  Sat  Sun  Thur  All
time   smoker                          
Dinner No        3   45   57     1  106
       Yes       9   42   19     0   70
Lunch  No        1    0    0    44   45
       Yes       6    0    0    17   23
All             19   87   76    62  244

> **🛠️ Exercise:** Compute the sum of `tip` in a pivot table with `day` and `time` in the rows and `smoker` in the column.

### Cross-Tabulation

A _cross-tabulation_ or _crosstab_ is a special case of pivot table that computes group frequencies (counts):

In [89]:
pd.crosstab(index=[tips["time"], tips["day"]], columns=tips["smoker"], margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244

---
# Part 4: Advanced Toolkit (Optional / Deep Dive)

**Learning Objective:** Master complex data structures and advanced transformations. This section covers Hierarchical Indexing, Stacking, Concatenation, and custom Apply/Transform logic.

## 4.1 Hierarchical Indexing (MultiIndex)

Hierarchical indexing (MultiIndex) allows you to have multiple (two or more) _index levels_ on an axis. It enables "higher dimensional" data in a lower dimensional data structure.

In [90]:
data = pd.Series(np.random.uniform(size=9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                 [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

a  1    0.628992
   2    0.254221
   3    0.307800
b  1    0.103733
   3    0.801026
c  1    0.625795
   2    0.498626
d  2    0.694515
   3    0.234360
dtype: float64

In [91]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

You can use _partial indexing_ to select subsets of data:

In [92]:
data["b"]

1    0.103733
3    0.801026
dtype: float64

In [93]:
data["b":"c"]

b  1    0.103733
   3    0.801026
c  1    0.625795
   2    0.498626
dtype: float64

In [94]:
data.loc[["b", "d"]]

b  1    0.103733
   3    0.801026
d  2    0.694515
   3    0.234360
dtype: float64

You can also select from "inner" level:

In [95]:
data.loc[:, 2]

a    0.254221
c    0.498626
d    0.694515
dtype: float64

Hierarchical indexing works on both axes.

In [96]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                        index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                        columns=[['Ohio', 'Ohio', 'Colorado'],
                        ['Green', 'Red', 'Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

Setting names on the axes work as usual:

In [97]:
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [102]:
frame.index.nlevels

2

Partial indexing works on columns too:

In [103]:
frame["Ohio"]

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

### Reordering and Sorting Levels

You may need to rearrange the order of the levels on an axis. The `swaplevel` method will swap the levels. The default is to swap the levels on the rows:

In [104]:
frame.swaplevel()

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [105]:
frame.swaplevel(0, 1, axis=1)

color     Green  Red    Green
state      Ohio Ohio Colorado
key1 key2                    
a    1        0    1        2
     2        3    4        5
b    1        6    7        8
     2        9   10       11

You can also sort by a single level or subset of levels:

In [106]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

> > **🛠️ Exercise:** Swap the levels on the rows then sort the index by level `0`.

### Setting and Resetting Index

It's common to use one or more columns from a DataFrame as the row index.

In [107]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1), 
                      "c": ["one", "one", "one", "two", "two", "two", "two"], 
                      "d": [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


`set_index` will return a new DataFrame using one or more of its columns as the index.

In [108]:
frame2 = frame.set_index(["c", "d"])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

`reset_index` does the opposite of `set_index` and turns the index back into a column.

In [109]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


You can choose to drop the columns when resetting index:

In [ ]:
frame2.reset_index(drop=True)

## 4.2 Concatenation

You can join DataFrames along any axis which is referred to as _concatenation_ or _stacking_. This is akin to database `union` operations.

In [ ]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")

In [ ]:
pd.concat([s1, s2, s3])

By default, `concat` works along `axis="index"`, producing another Series. If you pass `axis="columns"`, the result will instead be a DataFrame:

In [ ]:
pd.concat([s1, s2, s3], axis="columns")

The default behavior of `concat` is union (`outer` join) of the indexes, you can also intersect them by passing `join='inner'`:

In [ ]:
s4 = pd.concat([s1, s3])
pd.concat([s1, s4], axis="columns", join="inner")

When combining Series along axis="columns", pass the `keys` argument for the DataFrame column headers:

In [ ]:
pd.concat([s1, s2, s3], axis="columns", keys=["one", "two", "three"])

> **🛠️ Exercise:** Concat `s1`, `s2` and `s3` along index and pass `keys=["one", "two", "three"]`.

If the index does not contain any relevant data, and you want to avoid concatenating based on indexes, you can pass the `ignore_index=True` argument:

In [ ]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)), 
                   columns=["a", "b", "c", "d"])

df2 = pd.DataFrame(np.random.standard_normal((2, 3)), 
                   columns=["b", "d", "a"])

pd.concat([df1, df2], ignore_index=True)

## 4.3 Stacking and Unstacking

These are alternative reshaping methods to Melt/Pivot that work specifically on the Index levels.

In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)), 
                    index=pd.Index(["Ohio", "Colorado"], name="state"),
                    columns=pd.Index(["one", "two", "three"], name="number"))
data

The `stack` method pivots the columns into rows, producing a Series with a MultiIndex.

In [ ]:
result = data.stack()
result

From a hierarchically indexed Series, you can rearrange the data back into a DataFrame with `unstack` , which pivots rows into columns.

In [ ]:
result.unstack()

You can unstack a different level by passing a level number or name:

In [ ]:
result.unstack(level=0)

## 4.4 Advanced GroupBy: Apply

The most general-purpose GroupBy method is `apply`, which splits the object being manipulated into pieces, invokes the passed function on each piece, and then concatenates the pieces.

Suppose we want to select the top five `tip_pct` values by group. First, write a function that selects the rows with the largest values in a particular column:

In [ ]:
def top(df, n=5, column="tip_pct"):
    return df.sort_values(column, ascending=False)[:n]

In [ ]:
top(tips, n=6)

We can then `apply` this function by different groups using `groupby`:

In [ ]:
tips.groupby("smoker").apply(top)

You can pass the arguments to the function as follows:

In [ ]:
tips.groupby(["smoker", "day"]).apply(top, n=2, column="total_bill")

> **🛠️ Exercise:** Apply the function on `day` and `time` group.

## 4.5 Advanced GroupBy: Transform

You can also transform your data using the `transform` method. It is similar to `apply` but the function must:
- Produce a scalar value to be broadcast to the shape of the group chunk, or
- Return an object that is the same shape as the group chunk

This is useful for z-score normalization within groups.

In [ ]:
df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4, 'value': np.arange(12.)})
g = df.groupby('key')['value']
g.mean()

`transform` produce a Series of the same shape as `df['value']` but with values replaced by the average grouped by `key`.

In [ ]:
g.transform(lambda g: g.mean())

In [ ]:
def normalize(x):
    return (x - x.mean()) / x.std()

g.transform(normalize)